# 1. Cohort Preparation

Load registry data and apply inclusion/exclusion criteria for the LiverMets analysis.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('LiverMets_Final_Dataset.csv')
print(f"Raw registry: {len(df):,} patients × {df.shape[1]} variables")
print(f"\nColumns: {df.columns.tolist()}")

## Apply Inclusion/Exclusion Criteria

**Inclusion**: Complete TNM staging + survival data  
**Exclusion**: Missing T-stage, N-stage, M-stage, or survival information

In [ ]:
# Complete TNM
complete_tnm = df[
    (df['T_STAGE'].notna()) & (df['T_STAGE'] != 'ND') &
    (df['N_STAGE'].notna()) & (df['N_STAGE'] != 'ND') &
    (df['M_STAGE'].notna()) & (df['M_STAGE'] != 'ND')
]
print(f"Complete TNM: {len(complete_tnm):,} patients")
print(f"Excluded (missing TNM): {len(df) - len(complete_tnm):,}")

# Complete survival
included = complete_tnm[
    (complete_tnm['SURVIVAL_YEARS'].notna()) & 
    (complete_tnm['SURVIVAL_YEARS'] > 0) &
    (complete_tnm['VITAL_STATUS'].notna())
]
print(f"\nComplete survival: {len(included):,} patients")
print(f"Excluded (missing survival): {len(complete_tnm) - len(included):,}")

excluded = df[~df.index.isin(included.index)]
print(f"\nFinal cohorts:")
print(f"  Included:  {len(included):,} (49.9%)")
print(f"  Excluded:  {len(excluded):,} (50.1%)")
print(f"  Total:     {len(included) + len(excluded):,}")

## Data Quality Checks

In [ ]:
# Check key variables
print("Included cohort data completeness:")
print(f"  Age: {included['AGE_AT_REFERRAL'].notna().sum():,}/{len(included):,}")
print(f"  Gender: {included['GENDER'].notna().sum():,}/{len(included):,}")
print(f"  Treatment: {included['TREATMENT'].notna().sum():,}/{len(included):,}")
print(f"  Metastases: {included['NB_METS_GROUP'].notna().sum():,}/{len(included):,}")
print(f"  Follow-up: {included['SURVIVAL_YEARS'].notna().sum():,}/{len(included):,}")

# TNM distribution
print(f"\nT-stage distribution:")
print(included['T_STAGE'].value_counts().sort_index())

print(f"\nN-stage distribution:")
print(included['N_STAGE'].value_counts().sort_index())

print(f"\nM-stage distribution:")
print(included['M_STAGE'].value_counts())

## Outcome Summary

In [ ]:
deaths = (included['VITAL_STATUS'] == 1).sum()
alive = (included['VITAL_STATUS'] == 0).sum()

print(f"Mortality in included cohort:")
print(f"  Deaths: {deaths:,} ({100*deaths/len(included):.1f}%)")
print(f"  Censored/Alive: {alive:,} ({100*alive/len(included):.1f}%)")
print(f"\nFollow-up (years):")
print(f"  Mean: {included['SURVIVAL_YEARS'].mean():.2f} ± {included['SURVIVAL_YEARS'].std():.2f}")
print(f"  Median: {included['SURVIVAL_YEARS'].median():.2f}")
print(f"  Range: {included['SURVIVAL_YEARS'].min():.2f} – {included['SURVIVAL_YEARS'].max():.2f}")